<a href="https://colab.research.google.com/github/Anseekaa/Anseekaa/blob/main/PolySem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers datasets accelerate scikit-learn pandas numpy matplotlib seaborn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 10.7 MB/s  0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 2.4.0
    Uninstalling huggingface_hub-2.4.0:
      Successfully uninstalled huggingface_hub-2.4.0


In [ ]:
import pandas as pd
import numpy as np
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

print("Libraries imported successfully!")

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Libraries imported successfully!
GPU available: True
GPU: Tesla T4


In [4]:
from google.colab import files

uploaded = files.upload()

Saving polysem_test.csv to polysem_test (1).csv
Saving polysem_train.csv to polysem_train (1).csv
Saving polysem_validation.csv to polysem_validation (1).csv


In [ ]:
train_df = pd.read_csv("polysem_train.csv")
val_df = pd.read_csv("polysem_validation.csv")
test_df = pd.read_csv("polysem_test.csv")

print("Training dataset:", train_df.shape)
print("Validation dataset:", val_df.shape)
print("Testing dataset:", test_df.shape)

display(train_df.head())

In [ ]:
print("Training label distribution:")
print(train_df["label"].value_counts().sort_index())

print("\nValidation label distribution:")
print(val_df["label"].value_counts().sort_index())

print("\nTesting label distribution:")
print(test_df["label"].value_counts().sort_index())

print("\nMissing values:")
print(train_df.isnull().sum())

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "bert-base-multilingual-cased"


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3
)

print("mBERT tokenizer and model loaded successfully!")

In [ ]:
MAX_LENGTH = 128

def tokenize_data(df):
    return tokenizer(
        df["premise"].astype(str).tolist(),
        df["hypothesis"].astype(str).tolist(),
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH
    )

train_encodings = tokenize_data(train_df)
val_encodings = tokenize_data(val_df)
test_encodings = tokenize_data(test_df)

print("Tokenization completed!")

print("Training examples:", len(train_encodings["input_ids"]))
print("Validation examples:", len(val_encodings["input_ids"]))
print("Testing examples:", len(test_encodings["input_ids"]))

In [ ]:
import torch
from torch.utils.data import Dataset

class PolySemDataset(Dataset):

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }

        item["labels"] = torch.tensor(
            int(self.labels[idx]),
            dtype=torch.long
        )

        return item


train_dataset = PolySemDataset(
    train_encodings,
    train_df["label"].tolist()
)

val_dataset = PolySemDataset(
    val_encodings,
    val_df["label"].tolist()
)

test_dataset = PolySemDataset(
    test_encodings,
    test_df["label"].tolist()
)

print("PyTorch datasets created successfully!")
print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Testing samples:", len(test_dataset))

In [ ]:
sample = train_dataset[0]

print("Input IDs shape:", sample["input_ids"].shape)
print("Attention mask shape:", sample["attention_mask"].shape)
print("Label:", sample["labels"].item())

print("\nOriginal premise:")
print(train_df.iloc[0]["premise"])

print("\nOriginal hypothesis:")
print(train_df.iloc[0]["hypothesis"])

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="macro",
        zero_division=0
    )

    accuracy = accuracy_score(labels, predictions)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [ ]:
from transformers import TrainingArguments, Trainer
import torch

training_args = TrainingArguments(
    output_dir="./polysem_mbert_results",

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    num_train_epochs=5,
    weight_decay=0.01,

    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    save_total_limit=2,
    logging_steps=50,

    report_to="none",

    fp16=torch.cuda.is_available(),

    seed=42
)

print("Training configuration created!")

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

print("mBERT Trainer initialized successfully!")

In [ ]:
training_result = trainer.train()

print("mBERT training completed!")

In [ ]:
test_results = trainer.evaluate(test_dataset)

print("Final mBERT Test Results:")
for metric, value in test_results.items():
    if isinstance(value, (int, float)):
        print(f"{metric}: {value:.4f}")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

pred_output = trainer.predict(test_dataset)

y_true = pred_output.label_ids
y_pred = np.argmax(pred_output.predictions, axis=1)

print(classification_report(
    y_true,
    y_pred,
    target_names=["Entailment", "Neutral", "Contradiction"],
    digits=4
))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Entailment", "Neutral", "Contradiction"],
    yticklabels=["Entailment", "Neutral", "Contradiction"]
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("PolySem mBERT Confusion Matrix")
plt.show()

In [ ]:
model.config.id2label = {
    0: "Entailment",
    1: "Neutral",
    2: "Contradiction"
}

model.config.label2id = {
    "Entailment": 0,
    "Neutral": 1,
    "Contradiction": 2
}

trainer.save_model("./polysem_mbert_best")
tokenizer.save_pretrained("./polysem_mbert_best")

print("Best mBERT model saved!")

In [ ]:
!pip install gradio

In [ ]:
import gradio as gr
import torch
import numpy as np

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model.to(device)
model.eval()

labels = [
    "Entailment",
    "Neutral",
    "Contradiction"
]

def predict_relation(premise, hypothesis):

    if not premise.strip() or not hypothesis.strip():
        return "Please enter both sentences.", {}

    inputs = tokenizer(
        premise,
        hypothesis,
        return_tensors="pt",
        truncation=True,
        max_length=128
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        outputs = model(**inputs)

        probabilities = torch.softmax(
            outputs.logits,
            dim=1
        )[0]

    prediction = torch.argmax(probabilities).item()

    confidence = probabilities[prediction].item()

    scores = {
        labels[i]: float(probabilities[i])
        for i in range(3)
    }

    result = (
        f"Prediction: {labels[prediction]}\n"
        f"Confidence: {confidence:.2%}"
    )

    return result, scores

In [ ]:
with gr.Blocks(theme=gr.themes.Soft()) as demo:

    gr.Markdown(
        """
        # PolySem
        ### Cross-Lingual Semantic Understanding

        Analyze the relationship between two sentences.
        """
    )

    premise = gr.Textbox(
        label="Premise",
        placeholder="Enter the first sentence..."
    )

    hypothesis = gr.Textbox(
        label="Hypothesis",
        placeholder="Enter the second sentence..."
    )

    analyze_btn = gr.Button(
        "Analyze Meaning",
        variant="primary"
    )

    prediction = gr.Textbox(
        label="Predicted Relationship"
    )

    confidence_scores = gr.Label(
        label="Class Confidence Scores"
    )

    analyze_btn.click(
        fn=predict_relation,
        inputs=[premise, hypothesis],
        outputs=[prediction, confidence_scores]
    )

demo.launch(share=True)

In [ ]:
test_results = trainer.evaluate(test_dataset)

print("mBERT Test Accuracy:", test_results["eval_accuracy"])
print("mBERT Test Macro F1:", test_results["eval_f1"])

In [ ]:
trainer.save_model("./polysem_mbert_best")
tokenizer.save_pretrained("./polysem_mbert_best")

In [ ]:
import gradio as gr
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)
model.eval()

labels = ["Entailment", "Neutral", "Contradiction"]

def predict_relation(premise, hypothesis):

    if not premise.strip() or not hypothesis.strip():
        return "Please enter both sentences.", {}

    inputs = tokenizer(
        premise,
        hypothesis,
        return_tensors="pt",
        truncation=True,
        max_length=128
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        probabilities = torch.softmax(outputs.logits, dim=1)[0]

    prediction = probabilities.argmax().item()

    result = (
        f"Prediction: {labels[prediction]}\n"
        f"Confidence: {probabilities[prediction].item():.2%}"
    )

    scores = {
        labels[i]: float(probabilities[i])
        for i in range(3)
    }

    return result, scores

In [ ]:

with gr.Blocks(theme=gr.themes.Soft()) as demo:

    gr.Markdown("""
    # PolySem
    ### Cross-Lingual Semantic Understanding

    Analyze the relationship between two sentences.
    """)

    language_direction = gr.Radio(
        choices=[
            "English → Hindi",
            "Hindi → English",
            "English → English",
            "Hindi → Hindi"
        ],
        value="English → Hindi",
        label="Language Direction"
    )

    premise = gr.Textbox(
        label="Premise",
        placeholder="Enter the first sentence..."
    )

    hypothesis = gr.Textbox(
        label="Hypothesis",
        placeholder="Enter the second sentence..."
    )

    gr.Examples(
        examples=[
            [
                "A man is playing football.",
                "एक आदमी फुटबॉल खेल रहा है।"
            ],
            [
                "A man is playing football.",
                "A man is playing basketball."
            ],
            [
                "A man is playing football.",
                "A man is wearing a blue shirt."
            ]
        ],
        inputs=[premise, hypothesis]
    )

    analyze = gr.Button(
        "Analyze Meaning",
        variant="primary"
    )

    prediction = gr.Textbox(
        label="Predicted Relationship"
    )

    scores = gr.Label(
        label="Confidence Scores"
    )

    analyze.click(
        fn=predict_relation,
        inputs=[premise, hypothesis],
        outputs=[prediction, scores]
    )

demo.launch(share=True)


In [ ]:
test_examples = [
    (
        "A man is playing football.",
        "A man is playing a sport."
    ),
    (
        "A man is playing football.",
        "A man is playing basketball."
    ),
    (
        "A man is playing football.",
        "A man is wearing a blue shirt."
    ),
    (
        "A man is playing football.",
        "एक आदमी फुटबॉल खेल रहा है।"
    ),
    (
        "एक आदमी फुटबॉल खेल रहा है।",
        "A man is playing football."
    )
]

for premise, hypothesis in test_examples:

    prediction, scores = predict_relation(
        premise,
        hypothesis
    )

    print("\nPremise:", premise)
    print("Hypothesis:", hypothesis)
    print(prediction)
    print(scores)
    print("-" * 60)

In [ ]:
test_results = trainer.evaluate(test_dataset)

print("mBERT Test Accuracy:",
      round(test_results["eval_accuracy"] * 100, 2), "%")

print("mBERT Test Macro F1:",
      round(test_results["eval_f1"] * 100, 2), "%")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os

save_path = "/content/drive/MyDrive/PolySem"

os.makedirs(save_path, exist_ok=True)

trainer.save_model(f"{save_path}/mbert_model")
tokenizer.save_pretrained(f"{save_path}/mbert_model")

print("PolySem mBERT model saved to Google Drive!")

In [ ]:
train_df.to_csv(
    "/content/drive/MyDrive/PolySem/train.csv",
    index=False
)

val_df.to_csv(
    "/content/drive/MyDrive/PolySem/validation.csv",
    index=False
)

test_df.to_csv(
    "/content/drive/MyDrive/PolySem/test.csv",
    index=False
)

print("Datasets saved!")